### Comprehensive Python program for well performance classification and prediction using both deep learning and machine learning approaches. 
#### This program will handle oil, gas, and water production forecasting.

#### Keith R. Holdaway FGS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Data preprocessing and feature engineering
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

# Machine Learning models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Deep Learning models
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout, BatchNormalization, Input, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Time series specific
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

class WellPerformanceAnalyzer:
    def __init__(self):
        self.scalers = {}
        self.models = {}
        self.feature_importance = {}
        
    def generate_synthetic_data(self, n_wells=100, n_days=365*3):
        """Generate synthetic well production data for demonstration"""
        np.random.seed(42)
        
        data = []
        base_date = datetime(2020, 1, 1)
        
        for well_id in range(1, n_wells + 1):
            # Well characteristics
            well_type = np.random.choice(['Horizontal', 'Vertical'], p=[0.7, 0.3])
            reservoir_quality = np.random.uniform(0.5, 1.0)
            initial_pressure = np.random.uniform(2000, 5000)
            
            for day in range(n_days):
                date = base_date + timedelta(days=day)
                
                # Time-dependent factors
                time_factor = 1 / (1 + 0.001 * day)  # Decline factor
                seasonal_factor = 1 + 0.2 * np.sin(2 * np.pi * day / 365)  # Seasonal variation
                
                # Production calculations with noise
                oil_rate = (1000 * reservoir_quality * time_factor * seasonal_factor * 
                           (2 if well_type == 'Horizontal' else 1) * 
                           np.random.lognormal(0, 0.1))
                
                gas_rate = oil_rate * np.random.uniform(0.8, 1.2) * np.random.lognormal(0, 0.05)
                water_rate = oil_rate * np.random.uniform(0.1, 0.5) * np.random.lognormal(0, 0.08)
                
                data.append({
                    'well_id': well_id,
                    'date': date,
                    'well_type': well_type,
                    'reservoir_quality': reservoir_quality,
                    'initial_pressure': initial_pressure,
                    'days_online': day,
                    'oil_rate': oil_rate,
                    'gas_rate': gas_rate,
                    'water_rate': water_rate,
                    'water_cut': water_rate / (oil_rate + water_rate) if (oil_rate + water_rate) > 0 else 0,
                    'gor': gas_rate / oil_rate if oil_rate > 0 else 0
                })
        
        return pd.DataFrame(data)
    
    def create_features(self, df):
        """Create additional features for time series analysis"""
        df = df.copy()
        
        # Time-based features
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['day_of_year'] = df['date'].dt.dayofyear
        
        # Lag features
        for lag in [1, 7, 30, 90]:
            df[f'oil_lag_{lag}'] = df.groupby('well_id')['oil_rate'].shift(lag)
            df[f'gas_lag_{lag}'] = df.groupby('well_id')['gas_rate'].shift(lag)
            df[f'water_lag_{lag}'] = df.groupby('well_id')['water_rate'].shift(lag)
        
        # Rolling statistics
        for window in [7, 30, 90]:
            df[f'oil_ma_{window}'] = df.groupby('well_id')['oil_rate'].transform(
                lambda x: x.rolling(window=window, min_periods=1).mean())
            df[f'oil_std_{window}'] = df.groupby('well_id')['oil_rate'].transform(
                lambda x: x.rolling(window=window, min_periods=1).std())
        
        # Decline curve features
        df['decline_rate'] = df.groupby('well_id')['oil_rate'].transform(
            lambda x: -x.pct_change().rolling(30, min_periods=1).mean())
        
        return df.dropna()
    
    def prepare_data(self, df, target='oil_rate', forecast_horizon=30):
        """Prepare data for training and testing"""
        # Encode categorical variables
        le = LabelEncoder()
        df['well_type_encoded'] = le.fit_transform(df['well_type'])
        
        # Features and target
        feature_columns = [col for col in df.columns if col not in [
            'well_id', 'date', 'well_type', 'oil_rate', 'gas_rate', 'water_rate'
        ] and not col.startswith('target')]
        
        X = df[feature_columns].select_dtypes(include=[np.number])
        y = df[target]
        
        # Create future target for forecasting
        df[f'target_{forecast_horizon}'] = df.groupby('well_id')[target].shift(-forecast_horizon)
        y_future = df[f'target_{forecast_horizon}'].dropna()
        X_future = X.loc[y_future.index]
        
        return X, y, X_future, y_future, feature_columns
    
    def train_ml_models(self, X_train, y_train, X_test, y_test):
        """Train various machine learning models"""
        ml_models = {
            'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
            'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
            'SVR': SVR(kernel='rbf', C=1.0),
            'LinearRegression': LinearRegression(),
            'Ridge': Ridge(alpha=1.0),
            'Lasso': Lasso(alpha=0.1)
        }
        
        results = {}
        for name, model in ml_models.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            results[name] = {
                'model': model,
                'mae': mean_absolute_error(y_test, y_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                'r2': r2_score(y_test, y_pred)
            }
            
            # Feature importance for tree-based models
            if hasattr(model, 'feature_importances_'):
                self.feature_importance[name] = model.feature_importances_
        
        return results
    
    def create_lstm_model(self, input_shape):
        """Create LSTM model for time series forecasting"""
        model = Sequential([
            LSTM(128, return_sequences=True, input_shape=input_shape),
            Dropout(0.2),
            BatchNormalization(),
            LSTM(64, return_sequences=True),
            Dropout(0.2),
            BatchNormalization(),
            LSTM(32),
            Dropout(0.2),
            Dense(32, activation='relu'),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer=Adam(learning_rate=0.001),
                     loss='mse',
                     metrics=['mae'])
        return model
    
    def create_attention_model(self, input_shape):
        """Create model with attention mechanism"""
        inputs = Input(shape=input_shape)
        
        # LSTM layers
        lstm_out = LSTM(64, return_sequences=True)(inputs)
        lstm_out = Dropout(0.3)(lstm_out)
        lstm_out = BatchNormalization()(lstm_out)
        
        # Attention mechanism
        attention = Dense(1, activation='tanh')(lstm_out)
        attention = tf.keras.layers.Flatten()(attention)
        attention = tf.keras.layers.Activation('softmax')(attention)
        attention = tf.keras.layers.RepeatVector(64)(attention)
        attention = tf.keras.layers.Permute([2, 1])(attention)
        
        # Apply attention
        attended = tf.keras.layers.multiply([lstm_out, attention])
        attended = tf.keras.layers.Lambda(lambda x: tf.keras.backend.sum(x, axis=1))(attended)
        
        # Output layers
        x = Dense(32, activation='relu')(attended)
        x = Dropout(0.2)(x)
        x = Dense(16, activation='relu')(x)
        outputs = Dense(1)(x)
        
        model = Model(inputs, outputs)
        model.compile(optimizer=Adam(learning_rate=0.001),
                     loss='mse',
                     metrics=['mae'])
        return model
    
    def prepare_sequences(self, df, target='oil_rate', sequence_length=30):
        """Prepare sequences for LSTM training"""
        sequences = []
        targets = []
        
        for well_id in df['well_id'].unique():
            well_data = df[df['well_id'] == well_id].sort_values('date')
            well_values = well_data[target].values
            
            for i in range(len(well_values) - sequence_length):
                sequences.append(well_values[i:i + sequence_length])
                targets.append(well_values[i + sequence_length])
        
        return np.array(sequences), np.array(targets)
    
    def train_deep_learning_models(self, X_seq, y_seq, X_test_seq, y_test_seq):
        """Train deep learning models"""
        # Reshape for LSTM (samples, timesteps, features)
        X_seq = X_seq.reshape((X_seq.shape[0], X_seq.shape[1], 1))
        X_test_seq = X_test_seq.reshape((X_test_seq.shape[0], X_test_seq.shape[1], 1))
        
        # LSTM Model
        lstm_model = self.create_lstm_model((X_seq.shape[1], X_seq.shape[2]))
        
        # Attention Model
        attention_model = self.create_attention_model((X_seq.shape[1], X_seq.shape[2]))
        
        # Callbacks
        callbacks = [
            EarlyStopping(patience=10, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=5)
        ]
        
        # Train models
        print("Training LSTM model...")
        lstm_history = lstm_model.fit(
            X_seq, y_seq,
            epochs=50,
            batch_size=32,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )
        
        print("Training Attention model...")
        attention_history = attention_model.fit(
            X_seq, y_seq,
            epochs=50,
            batch_size=32,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate models
        lstm_pred = lstm_model.predict(X_test_seq)
        attention_pred = attention_model.predict(X_test_seq)
        
        results = {
            'LSTM': {
                'model': lstm_model,
                'history': lstm_history,
                'mae': mean_absolute_error(y_test_seq, lstm_pred),
                'rmse': np.sqrt(mean_squared_error(y_test_seq, lstm_pred)),
                'r2': r2_score(y_test_seq, lstm_pred)
            },
            'Attention': {
                'model': attention_model,
                'history': attention_history,
                'mae': mean_absolute_error(y_test_seq, attention_pred),
                'rmse': np.sqrt(mean_squared_error(y_test_seq, attention_pred)),
                'r2': r2_score(y_test_seq, attention_pred)
            }
        }
        
        return results
    
    def plot_results(self, ml_results, dl_results, actual, predictions, target_name):
        """Plot comparison of results"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Model comparison
        models = list(ml_results.keys()) + list(dl_results.keys())
        mae_scores = [ml_results[m]['mae'] for m in ml_results] + [dl_results[m]['mae'] for m in dl_results]
        
        axes[0, 0].bar(models, mae_scores)
        axes[0, 0].set_title('Model Comparison (MAE)')
        axes[0, 0].set_ylabel('Mean Absolute Error')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Actual vs Predicted
        axes[0, 1].scatter(actual, predictions, alpha=0.6)
        axes[0, 1].plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--')
        axes[0, 1].set_xlabel('Actual')
        axes[0, 1].set_ylabel('Predicted')
        axes[0, 1].set_title('Actual vs Predicted')
        
        # Time series plot
        axes[1, 0].plot(actual.values[:100], label='Actual', alpha=0.7)
        axes[1, 0].plot(predictions[:100], label='Predicted', alpha=0.7)
        axes[1, 0].set_title('Time Series Forecast')
        axes[1, 0].legend()
        
        # Feature importance
        if self.feature_importance:
            importance_df = pd.DataFrame(self.feature_importance)
            importance_df.mean(axis=1).sort_values().plot(kind='barh', ax=axes[1, 1])
            axes[1, 1].set_title('Feature Importance')
        
        plt.tight_layout()
        plt.savefig(f'well_performance_results_{target_name}.png')
        plt.show()
    
    def run_experiment(self, target='oil_rate', forecast_horizon=30):
        """Main experiment runner"""
        print(f"Running experiment for {target} forecasting...")
        
        # Generate and prepare data
        df = self.generate_synthetic_data()
        df = self.create_features(df)
        X, y, X_future, y_future, feature_columns = self.prepare_data(df, target, forecast_horizon)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X_future, y_future, test_size=0.2, random_state=42, shuffle=False
        )
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train ML models
        print("Training Machine Learning models...")
        ml_results = self.train_ml_models(X_train_scaled, y_train, X_test_scaled, y_test)
        
        # Prepare sequences for DL models
        X_seq, y_seq = self.prepare_sequences(df, target, sequence_length=30)
        X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
            X_seq, y_seq, test_size=0.2, random_state=42, shuffle=False
        )
        
        # Train DL models
        print("Training Deep Learning models...")
        dl_results = self.train_deep_learning_models(X_train_seq, y_train_seq, X_test_seq, y_test_seq)
        
        # Combine results
        best_ml_model = min(ml_results.items(), key=lambda x: x[1]['mae'])
        best_dl_model = min(dl_results.items(), key=lambda x: x[1]['mae'])
        
        print(f"\nBest ML Model: {best_ml_model[0]} - MAE: {best_ml_model[1]['mae']:.2f}")
        print(f"Best DL Model: {best_dl_model[0]} - MAE: {best_dl_model[1]['mae']:.2f}")
        
        # Make predictions with best model
        if best_ml_model[1]['mae'] < best_dl_model[1]['mae']:
            best_predictions = best_ml_model[1]['model'].predict(X_test_scaled)
            best_model_type = 'ML'
        else:
            best_predictions = best_dl_model[1]['model'].predict(
                X_test_seq.reshape((X_test_seq.shape[0], X_test_seq.shape[1], 1))
            ).flatten()
            best_model_type = 'DL'
        
        # Plot results
        self.plot_results(ml_results, dl_results, y_test, best_predictions, target)
        
        return {
            'ml_results': ml_results,
            'dl_results': dl_results,
            'best_model': best_ml_model if best_model_type == 'ML' else best_dl_model,
            'feature_importance': self.feature_importance,
            'predictions': best_predictions,
            'actual': y_test
        }

# Main execution
if __name__ == "__main__":
    analyzer = WellPerformanceAnalyzer()
    
    # Run experiments for different targets
    targets = ['oil_rate', 'gas_rate', 'water_rate']
    results = {}
    
    for target in targets:
        print(f"\n{'='*50}")
        print(f"EXPERIMENT: {target.upper()} FORECASTING")
        print(f"{'='*50}")
        
        result = analyzer.run_experiment(target=target, forecast_horizon=30)
        results[target] = result
        
        # Print summary
        print(f"\nSummary for {target}:")
        print(f"Best Model: {result['best_model'][0]}")
        print(f"MAE: {result['best_model'][1]['mae']:.2f}")
        print(f"RMSE: {result['best_model'][1]['rmse']:.2f}")
        print(f"R²: {result['best_model'][1]['r2']:.3f}")
    
    # Compare performance across targets
    print(f"\n{'='*50}")
    print("OVERALL PERFORMANCE COMPARISON")
    print(f"{'='*50}")
    
    # Create comparison dataframe
    comparison_data = {}
    for target in targets:
        comparison_data[target] = {
            'Best Model': results[target]['best_model'][0],
            'MAE': results[target]['best_model'][1]['mae'],
            'RMSE': results[target]['best_model'][1]['rmse'],
            'R²': results[target]['best_model'][1]['r2']
        }
    
    comparison_df = pd.DataFrame(comparison_data).T
    print(comparison_df)

### Key Features:
Synthetic Data Generation: Creates realistic oil, gas, and water production data with decline curves and seasonal variations

#### Feature Engineering:

Time-based features (lags, moving averages, decline rates)

Well characteristics (type, reservoir quality)

Statistical features

#### Machine Learning Models:

Random Forest

Gradient Boosting

Support Vector Regression

Linear models (Ridge, Lasso)

#### Deep Learning Models:

LSTM networks for sequence prediction

Attention mechanisms for improved performance

Custom neural network architectures

#### Comprehensive Evaluation:

Multiple error metrics (MAE, RMSE, R²)

Visualization of results

Feature importance analysis

Model comparison across targets

In [ ]:
# Initialize analyzer
# analyzer = WellPerformanceAnalyzer()

# Run experiment for oil production forecasting
# results = analyzer.run_experiment(target='oil_rate', forecast_horizon=30)

# Run for all targets
# targets = ['oil_rate', 'gas_rate', 'water_rate']
# for target in targets:
    # analyzer.run_experiment(target=target)

### Outputs:
Model performance comparisons

Time series forecasts

Feature importance plots

Statistical analysis of predictions

Visualizations of actual vs predicted values

### The program is designed to be modular and extensible, allowing you to easily add new models, features, or experiment with different hyperparameters for oil and gas production forecasting.